<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/15_review_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q anthropic pydantic tenacity sentence-transformers biopython
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, time, numpy as np, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
lib = pd.read_parquet(f'{BASE}/data/processed/library.parquet')
emb = np.load(f'{BASE}/data/processed/library_embeddings.npy')
cfg = json.load(open(f'{BASE}/data/processed/library_config.json'))
shortlist = pd.read_csv(f'{BASE}/results/shortlist.csv')
scores = pd.read_parquet(f'{BASE}/results/hallmark_scores_combined.parquet')
HALLMARKS = [c[:-4] for c in scores.columns if c.endswith('_pct')]
os.makedirs(f'{BASE}/results/dossiers_test', exist_ok=True)

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(cfg['embedding_model'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.0 MB/s eta 0:00:00
Mounted at /content/drive


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
def get_evidence(drug, hallmark, lib_, emb_, per_question=4, max_papers=10):
    rows = np.where(lib_['drug'].values == drug)[0]
    if len(rows) == 0:
        return []
    name = drug.replace('-', ' ')
    picked = []
    for template in cfg['questions'].values():
        q = embedder.encode([template.format(drug=name, hallmark=cfg['hallmark_text'][hallmark])],
                            normalize_embeddings=True)[0]
        for i in rows[np.argsort(-(emb_[rows] @ q))[:per_question]]:
            if i not in picked:
                picked.append(i)
    return [{'pmid': lib_.iloc[i]['pmid'], 'year': lib_.iloc[i]['year'], 'title': lib_.iloc[i]['title'],
             'abstract': lib_.iloc[i]['abstract'][:1500]} for i in picked[:max_papers]]

In [3]:
from pydantic import BaseModel
from typing import Literal

class AgingEvidence(BaseModel):
    cell: str
    animal: str
    human: str

class Dossier(BaseModel):
    drug: str
    hallmark: str
    mechanism: str
    aging_evidence: AgingEvidence
    safety: str
    confidence: Literal['High', 'Medium', 'Low']
    confidence_rationale: str
    stress_concern: bool
    key_refs: list[str]

In [4]:
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-5'   # the same model you used in Sessions 4, 7, and 8
PROMPT_VERSION = 'v1'

SYSTEM = """You are a skeptical aging-biology reviewer. You judge whether an existing drug might help one hallmark
of aging, based ONLY on the abstracts provided and the drug's screening score. Rules:
- Use only the abstracts given. If they do not address something, write "Not reported in the provided abstracts."
- A high screening score alone is weak evidence. With no relevant abstracts, confidence must be Low.
- High confidence requires direct evidence of a helpful effect on this hallmark, ideally in animals or people.
  Medium requires direct evidence in cells, or strong indirect evidence. Otherwise, Low.
- Set stress_concern to true if the evidence suggests the drug is toxic to cells or causes broad cellular stress,
  which can mimic a helpful gene-activity pattern.
- The screening data come from VCaP prostate cancer cells; keep that limitation in mind.
- Cite only PMIDs from the abstracts provided, in key_refs.
Respond with only a JSON object with keys: drug, hallmark, mechanism, aging_evidence (an object with keys cell,
animal, human), safety, confidence (High, Medium, or Low), confidence_rationale, stress_concern (true or false),
key_refs (a list of PMIDs)."""

def score_context(drug, hallmark):
    s = scores.loc[drug]
    others = [cfg['hallmark_text'][h] for h in HALLMARKS if h != hallmark and s[f'{h}_pct'] >= 95]
    return (f"Screening score for {cfg['hallmark_text'][hallmark]}: {s[f'{hallmark}_pct']:.0f}th percentile of "
            f"15,542 compounds (method: {s[f'{hallmark}_source']}). Also in the top 5% for: "
            f"{', '.join(others) or 'no other hallmarks'}.")

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def review(drug, hallmark, evidence):
    text = '\n\n'.join(f"PMID {a['pmid']} ({a['year']}): {a['title']}\n{a['abstract']}" for a in evidence)
    prompt = (f"Drug: {drug}\nHallmark: {cfg['hallmark_text'][hallmark]}\n{score_context(drug, hallmark)}\n\n"
              f"Abstracts:\n{text or 'No abstracts found.'}")
    msg = client.messages.create(model=MODEL, max_tokens=8000, system=SYSTEM,
                                 messages=[{'role': 'user', 'content': prompt}])
    raw = next(b.text for b in msg.content if b.type == 'text')
    return Dossier.model_validate_json(raw[raw.find('{'): raw.rfind('}') + 1])

In [5]:
counts = lib.groupby('drug').size()
sl = shortlist.set_index('drug')
first_hm = lambda d: sl.loc[d, 'hallmarks'].split(', ')[0]

tests = [sl.index[0], counts.idxmax(), sl.index[len(sl) // 2]]
few = counts[counts <= 5]
if len(few):
    tests.append(few.index[0])
no_papers = [d for d in sl.index if d not in counts.index]
if no_papers:
    tests.append(no_papers[0])
test_pairs = [(d, first_hm(d)) for d in dict.fromkeys(tests)]
print(test_pairs)

[('sj-172550', 'senescence'), ('amsacrine', 'nutrient_sensing'), ('16,16-dimethylprostaglandin-e2', 'mitochondria'), ('amcinonide', 'senescence'), ('ku-c103655', 'senescence')]


In [6]:
from Bio import Entrez
Entrez.email = 'your.email@example.com'   # your own email, as before
TOPICS = ('(aging OR ageing OR senescence OR lifespan OR longevity OR mTOR OR autophagy OR mitochondria '
          'OR mitochondrial OR inflammation OR "protein folding" OR "unfolded protein" OR toxicity OR mechanism)')

def fetch(drug, n=40):
    q = f'"{drug.replace("-", " ")}"[Title/Abstract] AND {TOPICS}'
    ids = Entrez.read(Entrez.esearch(db='pubmed', term=q, retmax=n, sort='relevance'))['IdList']
    recs = Entrez.read(Entrez.efetch(db='pubmed', id=','.join(ids), retmode='xml'))
    out = []
    for art in recs['PubmedArticle']:
        a = art['MedlineCitation']['Article']
        abstract = ' '.join(str(x) for x in a.get('Abstract', {}).get('AbstractText', []))
        if abstract:
            year = a.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {}).get('Year', '')
            out.append({'pmid': str(art['MedlineCitation']['PMID']), 'title': str(a['ArticleTitle']),
                        'year': str(year), 'abstract': abstract})
    return out

controls = [('metformin', 'nutrient_sensing'), ('tunicamycin', 'proteostasis')]
ctrl_lib = pd.DataFrame([{'drug': d, **a} for d, _ in controls for a in fetch(d)])
ctrl_emb = embedder.encode((ctrl_lib['title'] + '. ' + ctrl_lib['abstract']).tolist(), normalize_embeddings=True)
print(ctrl_lib.groupby('drug').size())

drug
metformin      40
tunicamycin    40
dtype: int64


In [7]:
def run(pairs, lib_, emb_):
    out = []
    for d, hm in pairs:
        path = f'{BASE}/results/dossiers_test/{PROMPT_VERSION}_{d}_{hm}.json'
        if os.path.exists(path):
            dz = json.load(open(path))
        else:
            ev = get_evidence(d, hm, lib_, emb_)
            dz = review(d, hm, ev).model_dump()
            dz['supplied_pmids'] = [a['pmid'] for a in ev]
            json.dump(dz, open(path, 'w'), indent=2)
        bad = [p for p in dz['key_refs'] if p not in dz['supplied_pmids']]
        print(f"{d:22} {hm:17} {len(dz['supplied_pmids']):2} papers -> {dz['confidence']:6} "
              f"stress={str(dz['stress_concern']):5} {'INVENTED PMIDs: ' + str(bad) if bad else 'citations OK'}")
        out.append(dz)
    return out

test_out = run(test_pairs, lib, emb)
ctrl_out = run(controls, ctrl_lib, ctrl_emb)

sj-172550              senescence         5 papers -> Low    stress=True  citations OK
amsacrine              nutrient_sensing   6 papers -> Low    stress=True  citations OK
16,16-dimethylprostaglandin-e2 mitochondria       6 papers -> Low    stress=True  citations OK
amcinonide             senescence         4 papers -> Low    stress=True  citations OK
ku-c103655             senescence         0 papers -> Low    stress=True  citations OK
metformin              nutrient_sensing   7 papers -> Medium stress=False citations OK
tunicamycin            proteostasis       9 papers -> Low    stress=True  citations OK


In [8]:
for dz in test_out + ctrl_out:
    print('=' * 100)
    print(json.dumps({k: v for k, v in dz.items() if k != 'supplied_pmids'}, indent=2))

{
  "drug": "SJ-172550",
  "hallmark": "cellular senescence",
  "mechanism": "SJ-172550 was identified as a small-molecule inhibitor of the MDMX\u2013p53 interaction, intended to restore p53 activity in tumors. However, subsequent studies show it acts through a complex, non-selective covalent-but-reversible mechanism, is chemically unstable in aqueous buffer, and binds promiscuously to cellular proteins rather than specifically stabilizing or inhibiting MDMX. Its described cellular effects are cytotoxic (killing retinoblastoma and other cancer cells) rather than related to modulating senescence.",
  "aging_evidence": {
    "cell": "Not reported in the provided abstracts. The compound kills cancer cells (retinoblastoma, MDA-MB-231, SK-N-MC) via cytotoxic/apoptotic or autophagic mechanisms in cancer cell lines, but no data address senescence induction, senescence-associated secretory phenotype, or clearance of senescent cells.",
    "animal": "Not reported in the provided abstracts.",
  

In [9]:
json.dump({'version': PROMPT_VERSION, 'model': MODEL, 'system': SYSTEM},
          open(f'{BASE}/data/processed/review_prompt.json', 'w'), indent=2)

conf = pd.Series([d['confidence'] for d in test_out]).value_counts().to_dict()
print(f"Designed a RAG review prompt (version {PROMPT_VERSION}): each review retrieves up to 10 relevant abstracts "
      f"from the library across mechanism, aging, and safety questions, and returns a schema-validated dossier with a "
      f"High/Medium/Low confidence rating and a cellular-stress flag. Tested on {len(test_out)} shortlisted drugs "
      f"({conf}) plus two calibration drugs: metformin rated {ctrl_out[0]['confidence']} on nutrient sensing, "
      f"tunicamycin rated {ctrl_out[1]['confidence']} (stress flag {ctrl_out[1]['stress_concern']}) on protein upkeep.")

Designed a RAG review prompt (version v1): each review retrieves up to 10 relevant abstracts from the library across mechanism, aging, and safety questions, and returns a schema-validated dossier with a High/Medium/Low confidence rating and a cellular-stress flag. Tested on 5 shortlisted drugs ({'Low': 5}) plus two calibration drugs: metformin rated Medium on nutrient sensing, tunicamycin rated Low (stress flag True) on protein upkeep.
